In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import os
import time
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet152V2
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix, recall_score, fbeta_score

In [3]:
TRAIN_PATH = "/content/drive/MyDrive/Cataract/Data/Train"
TEST_PATH  = "/content/drive/MyDrive/Cataract/Data/Test"

MODEL_PATH = "/content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5"

In [4]:
print("Train folders:", os.listdir(TRAIN_PATH))
print("Test folders:", os.listdir(TEST_PATH))

Train folders: ['Cataract', 'Normal', 'Not Eye']
Test folders: ['Cataract', 'Normal', 'Not Eye']


In [5]:
datagen = ImageDataGenerator(
    rescale = 1. / 255,
    validation_split = 0.2,
    horizontal_flip = True,
    vertical_flip = True
)

In [6]:
train_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'categorical',
    batch_size = 32,
    subset = "training"
)

val_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'categorical',
    batch_size = 32,
    subset = "validation"
)

test_it = datagen.flow_from_directory(
    TEST_PATH,
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'categorical',
    batch_size = 32,
    shuffle = False
)

print("Class indices:", train_it.class_indices)

Found 8896 images belonging to 3 classes.
Found 2221 images belonging to 3 classes.
Found 2552 images belonging to 3 classes.
Class indices: {'Cataract': 0, 'Normal': 1, 'Not Eye': 2}


In [7]:
base_model = ResNet152V2(
    weights = 'imagenet',
    input_shape = (224, 224, 3),
    include_top = False
)

234545216/234545216 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [8]:
ocl1 = Conv2D(32, (3, 3), activation='relu')(base_model.output)
bn1 = BatchNormalization()(ocl1)
mp1 = MaxPooling2D(pool_size=(2, 2))(bn1)
do1 = Dropout(0.17)(mp1)

ocl2 = Conv2D(64, (2, 2), activation='relu')(do1)
bn2 = BatchNormalization()(ocl2)

al1 = GlobalAveragePooling2D()(bn2)

fc1 = Dense(64, activation='relu')(al1)
fc2 = Dense(32, activation='relu')(fc1)
fc3 = Dense(32, activation='relu')(fc2)

al2 = BatchNormalization()(fc3)
all2 = Dropout(0.3)(al2)

output = Dense(3, activation='softmax', name='preds')(all2)

In [9]:
Cataract_Model = Model(
    inputs = base_model.input,
    outputs = output
)

Cataract_Model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_conv[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, 56, 56,    │        256 │ pool1_pool[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, 56, 56,    │          0 │ conv2_block1_pre… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,096 │ conv2_block1_pre… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_pad  │ (None, 58, 58,    │          0 │ conv2_block1_1_r… │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_2_p… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_pre… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_out    │ (None, 56, 56,    │          0 │ conv2_block1_0_c

 Total params: 58,937,667 (224.83 MB)

 Trainable params: 58,793,667 (224.28 MB)

 Non-trainable params: 144,000 (562.50 KB)

In [10]:
Cataract_Model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
for ix in range(564):
    Cataract_Model.layers[ix].trainable = False

In [ ]:
for i, layer in enumerate(Cataract_Model.layers):
    print(i, layer.name, layer.trainable)

0 input_layer False
1 conv1_pad False
2 conv1_conv False
3 pool1_pad False
4 pool1_pool False
5 conv2_block1_preact_bn False
6 conv2_block1_preact_relu False
7 conv2_block1_1_conv False
8 conv2_block1_1_bn False
9 conv2_block1_1_relu False
10 conv2_block1_2_pad False
11 conv2_block1_2_conv False
12 conv2_block1_2_bn False
13 conv2_block1_2_relu False
14 conv2_block1_0_conv False
15 conv2_block1_3_conv False
16 conv2_block1_out False
17 conv2_block2_preact_bn False
18 conv2_block2_preact_relu False
19 conv2_block2_1_conv False
20 conv2_block2_1_bn False
21 conv2_block2_1_relu False
22 conv2_block2_2_pad False
23 conv2_block2_2_conv False
24 conv2_block2_2_bn False
25 conv2_block2_2_relu False
26 conv2_block2_3_conv False
27 conv2_block2_out False
28 conv2_block3_preact_bn False
29 conv2_block3_preact_relu False
30 conv2_block3_1_conv False
31 conv2_block3_1_bn False
32 conv2_block3_1_relu False
33 conv2_block3_2_pad False
34 conv2_block3_2_conv False
35 conv2_block3_2_bn False
36 conv2_

In [ ]:
mc = ModelCheckpoint(
    MODEL_PATH,
    monitor = 'val_accuracy',
    mode = 'max',
    verbose = 1,
    save_best_only = True
)

In [ ]:
history = Cataract_Model.fit(
    train_it,
    epochs = 20,
    validation_data = val_it,
    callbacks = [mc]
)

Epoch 1/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 20s/step - accuracy: 0.6447 - loss: 0.8091 
Epoch 1: val_accuracy improved from None to 0.92301, saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 7066s 25s/step - accuracy: 0.7629 - loss: 0.5585 - val_accuracy: 0.9230 - val_loss: 0.2652
Epoch 2/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - accuracy: 0.9006 - loss: 0.2934
Epoch 2: val_accuracy improved from 0.92301 to 0.95813, saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 79s 283ms/step - accuracy: 0.9209 - loss: 0.2496 - val_accuracy: 0.9581 - val_loss: 0.1268
Epoch 3/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.9467 - loss: 0.1756
Epoch 3: val_accuracy improved from 0.95813 to 0.96803, saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5



Epoch 3: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 82s 294ms/step - accuracy: 0.9511 - loss: 0.1590 - val_accuracy: 0.9680 - val_loss: 0.0944
Epoch 4/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - accuracy: 0.9600 - loss: 0.1319
Epoch 4: val_accuracy improved from 0.96803 to 0.97929, saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5



Epoch 4: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 137s 279ms/step - accuracy: 0.9631 - loss: 0.1240 - val_accuracy: 0.9793 - val_loss: 0.0698
Epoch 5/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.9728 - loss: 0.1014
Epoch 5: val_accuracy improved from 0.97929 to 0.98694, saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5


...


Epoch 8: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 76s 273ms/step - accuracy: 0.9845 - loss: 0.0605 - val_accuracy: 0.9874 - val_loss: 0.0408
Epoch 9/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.9857 - loss: 0.0516
Epoch 9: val_accuracy improved from 0.98739 to 0.99145, saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5



Epoch 9: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 81s 290ms/step - accuracy: 0.9862 - loss: 0.0478 - val_accuracy: 0.9914 - val_loss: 0.0296
Epoch 10/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.9866 - loss: 0.0430
Epoch 10: val_accuracy improved from 0.99145 to 0.99235, saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5



Epoch 10: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 82s 293ms/step - accuracy: 0.9877 - loss: 0.0421 - val_accuracy: 0.9923 - val_loss: 0.0231
Epoch 11/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step - accuracy: 0.9903 - loss: 0.0372
Epoch 11: val_accuracy did not improve from 0.99235
278/278 ━━━━━━━━━━━━━━━━━━━━ 74s 264ms/step - accuracy: 0.9893 - loss: 0.0390 - val_accuracy: 0.9883 - val_loss: 0.0265
Epoch 12/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.9880 - loss: 0.0395
Epoch 12: val_accuracy did not improve from 0.99235
278/278 ━━━━━━━━━━━━━━━━━━━━ 70s 253ms/step - accuracy: 0.9882 - loss: 0.0364 - val_accuracy: 0.9910 - val_loss: 0.0242
Epoch 13/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.9888 - loss: 0.0378
Epoch 13: val_accuracy did not improve from 0.99235
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 254ms/step - accuracy: 0.9899 - loss: 0.0344 - val_accuracy: 0.9869 - val_loss: 0.0385
Epoch


Epoch 15: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 82s 293ms/step - accuracy: 0.9909 - loss: 0.0326 - val_accuracy: 0.9932 - val_loss: 0.0197
Epoch 16/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.9936 - loss: 0.0227
Epoch 16: val_accuracy did not improve from 0.99325
278/278 ━━━━━━━━━━━━━━━━━━━━ 73s 262ms/step - accuracy: 0.9931 - loss: 0.0250 - val_accuracy: 0.9932 - val_loss: 0.0188
Epoch 17/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step - accuracy: 0.9930 - loss: 0.0250
Epoch 17: val_accuracy did not improve from 0.99325
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 255ms/step - accuracy: 0.9920 - loss: 0.0259 - val_accuracy: 0.9928 - val_loss: 0.0220
Epoch 18/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.9906 - loss: 0.0320
Epoch 18: val_accuracy did not improve from 0.99325
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 255ms/step - accuracy: 0.9921 - loss: 0.0282 - val_accuracy: 0.9928 - val_loss: 0.0279
Epoch


Epoch 20: finished saving model to /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 77s 276ms/step - accuracy: 0.9955 - loss: 0.0160 - val_accuracy: 0.9941 - val_loss: 0.0144


In [11]:
best_model = tf.keras.models.load_model(MODEL_PATH)
print("Best model loaded:", MODEL_PATH)

Best model loaded: /content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5


In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_it)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

80/80 ━━━━━━━━━━━━━━━━━━━━ 898s 11s/step - accuracy: 0.9941 - loss: 0.0153
Test Loss: 0.015270201489329338
Test Accuracy: 0.9941222667694092


In [ ]:
test_it.reset()

y_pred_probs = best_model.predict(test_it)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_it.classes

recall = recall_score(y_true, y_pred, average='weighted')
f2 = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Recall:", recall)
print("F2 Score:", f2)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(test_it.class_indices.keys())))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

80/80 ━━━━━━━━━━━━━━━━━━━━ 40s 372ms/step
Recall: 0.9952978056426333
F2 Score: 0.9952968724326227

Classification Report:
              precision    recall  f1-score   support

    Cataract       0.99      0.99      0.99       800
      Normal       0.99      1.00      0.99       800
     Not Eye       1.00      1.00      1.00       952

    accuracy                           1.00      2552
   macro avg       1.00      1.00      1.00      2552
weighted avg       1.00      1.00      1.00      2552


Confusion Matrix:
[[792   8   0]
 [  3 797   0]
 [  1   0 951]]


In [15]:
import time
import numpy as np
import os

test_it.reset()
single_image_input = next(iter(test_it))[0][:1]
print(f"Input shape: {single_image_input.shape}  ← batch_size=1 (single image)")

print("Running warm-up inferences...")
for _ in range(10):
    best_model.predict(single_image_input, verbose=0)

N = 100
times = []
for _ in range(N):
    start = time.perf_counter()
    best_model.predict(single_image_input, verbose=0)
    end   = time.perf_counter()
    times.append((end - start) * 1000)

# ── Save 100 values to Drive ──────────────────────────────
np.save("/content/drive/MyDrive/Cataract/latency_ResNet152V2.npy", np.array(times))
print("✅ Saved latency_ResNet152V2.npy")

latency_mean = np.mean(times)
latency_std  = np.std(times)
latency_p95  = np.percentile(times, 95)
model_size_mb = os.path.getsize("/content/drive/MyDrive/Cataract/ResNet152V2_FineTune.h5") / (1024*1024)

print("\n" + "="*55)
print("  ResNet152V2 — Single-Image Latency Report")
print("="*55)
print(f"  Average Latency : {latency_mean:.2f} ± {latency_std:.2f} ms")
print(f"  P95 Latency     : {latency_p95:.2f} ms")
print(f"  Model File Size : {model_size_mb:.2f} MB")
print(f"  Benchmark Runs  : {N}")
print(f"  Input Shape     : {single_image_input.shape}")
print(f"  Hardware        : Google Colab T4 GPU")
print("="*55)

Input shape: (1, 224, 224, 3)  ← batch_size=1 (single image)
Running warm-up inferences...
✅ Saved latency_ResNet152V2.npy

  ResNet152V2 — Single-Image Latency Report
  Average Latency : 106.79 ± 21.03 ms
  P95 Latency     : 145.77 ms
  Model File Size : 231.14 MB
  Benchmark Runs  : 100
  Input Shape     : (1, 224, 224, 3)
  Hardware        : Google Colab T4 GPU
